# Logistic Regression – Airline Customer Satisfaction Prediction
### Project Overview & Machine Learning Pipeline
This notebook delivers an end-to-end classification analysis to predict customer satisfaction based on passenger survey metrics from *Invistico Airline*. Through data preprocessing, feature scaling, and binomial logistic regression, we uncover the critical drivers of customer retention and evaluate true-positive vs. false-positive performance trade-offs.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, classification_report, roc_auc_score

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
print('Libraries successfully imported.')

Libraries successfully imported.


## 1. Load the Dataset & Inspect Target Variable
We begin by importing the raw survey records and reviewing structural properties, class balance, and missing data.

In [2]:
df = pd.read_csv('Invistico_Airline.csv')
print(f'Dataset Shape: {df.shape}')
print('\nTarget Variable Distribution:')
print(df['satisfaction'].value_counts(dropna=False))
print('\nMissing Values Breakdown:')
print(df.isnull().sum()[df.isnull().sum() > 0])

Dataset Shape: (129880, 22)

Target Variable Distribution:
satisfaction
satisfied       71087
dissatisfied    58793

Missing Values Breakdown:
Arrival Delay in Minutes    393


## 2. Feature Engineering & Categorical Encoding
We map categorical variables into formats suitable for Scikit-learn models:
- Target variable `satisfaction`: Binary numeric (1 for satisfied, 0 for dissatisfied)
- Demographic/Travel categories mapped to binary or dummy encoded metrics.
- Impute missing entries in `Arrival Delay in Minutes` using its median value.

In [3]:
# Impute missing arrival delays with median
df['Arrival Delay in Minutes'] = df['Arrival Delay in Minutes'].fillna(df['Arrival Delay in Minutes'].median())

# Encode Binary Categoricals
df['satisfaction'] = df['satisfaction'].map({'satisfied': 1, 'dissatisfied': 0})
df['Customer Type'] = df['Customer Type'].map({'Loyal Customer': 1, 'disloyal Customer': 0})
df['Type of Travel'] = df['Type of Travel'].map({'Business travel': 1, 'Personal Travel': 0})

# One-Hot Encode nominal multiclass variables (Class)
df = pd.get_dummies(df, columns=['Class'], drop_first=True, dtype=int)
print('Preprocessing and encoding complete.')

## 3. Data Splitting & Feature Scaling
We split our target variable and feature space into an 80/20 train/test configuration with stratification. Standardization scales numeric attributes to ensure interpretable regression coefficients.

In [4]:
X = df.drop(columns=['satisfaction'])
y = df['satisfaction']

# Stratified Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Scale the features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f'Train size: {X_train_scaled.shape[0]}, Test size: {X_test_scaled.shape[0]}')

Train size: 103904, Test size: 25976


## 4. Model Training: Binomial Logistic Regression
We fit our model to the training data using the standard L2 penalized objective function.

In [5]:
model = LogisticRegression(max_iter=1000, random_state=42)
model.fit(X_train_scaled, y_train)
print('Model training complete.')

Model training complete.


## 5. Model Evaluation & Performance Analytics
We evaluate predictions beyond standard accuracy, inspecting precision, recall, and the confusion matrix matrix.

In [6]:
y_pred = model.predict(X_test_scaled)
y_prob = model.predict_proba(X_test_scaled)[:, 1]

print('--- CLASSIFICATION REPORT ---')
print(classification_report(y_test, y_pred))
print('Confusion Matrix:')
print(confusion_matrix(y_test, y_pred))

--- CLASSIFICATION REPORT ---
              precision    recall  f1-score   support

           0       0.81      0.81      0.81     11759
           1       0.84      0.84      0.84     14217

    accuracy                           0.83     25976
   macro avg       0.83      0.83      0.83     25976
weighted avg       0.83      0.83      0.83     25976

Confusion Matrix:
[[ 9546  2213]
 [ 2232 11985]]


## 6. Interpret Coefficients & Formulate Actionable Recommendations
By inspecting the model coefficients, we isolate the highest-impact factors influencing passenger satisfaction levels.

In [7]:
coef_df = pd.DataFrame({
    'Feature': X.columns,
    'Coefficient': model.coef_[0]
}).sort_values(by='Coefficient', ascending=False)

print('--- FEATURE COEFFICIENTS ---')
print(coef_df.to_string(index=False))

--- FEATURE COEFFICIENTS ---
                          Feature  Coefficient
           Inflight entertainment     0.972905
                    Customer Type     0.725012
                     Seat comfort     0.392582
                 On-board service     0.387346
                  Checkin service     0.355612
                   Type of Travel     0.349626
           Ease of Online booking     0.334382
                 Leg room service     0.308623
                  Online boarding     0.181621
                    Gate location     0.162930
                   Online support     0.144692
                 Baggage handling     0.109271
       Departure Delay in Minutes     0.073041
                      Cleanliness     0.064025
            Inflight wifi service    -0.127214
                              Age    -0.132652
                  Flight Distance    -0.181013
                   Class_Eco Plus    -0.197960
         Arrival Delay in Minutes    -0.261873
                   Food and dri